In [1]:
# %%
import pandas as pd
from pathlib import Path

# Rutas
ruta_cupratherm = Path(r"E:\ProyectoAnalisisElectrico\CupraThermV2\Resumen_Establecimientos.csv")
ruta_clienteslibres = Path(r"E:\ProyectoAnalisisElectrico\DiaPromedio\Periodos\2505_2604\2505_2604_mean_period_loc.parquet")

# Cargar las bases
print("Cargando bases de datos...")
ct = pd.read_csv(ruta_cupratherm, sep=",")
df = pd.read_parquet(ruta_clienteslibres)

print(f"Filas en Cupratherm: {len(ct)}")
print(f"Filas en Clientes Libres: {len(df)}")

Cargando bases de datos...
Filas en Cupratherm: 847
Filas en Clientes Libres: 163056


In [2]:
# %%
print("Estandarizando RUTs al formato numérico base...")

# 1. Clientes Libres: Extraer todo antes del guion y quitar puntos
df['rut_base'] = df['RUT'].astype(str).str.split('-').str[0].str.replace('.', '', regex=False).str.strip()

# 2. Cupratherm: Extraer todo antes del guion y quitar puntos
ct['rut_base'] = ct['RUT_RAZON_SOCIAL'].astype(str).str.split('-').str[0].str.replace('.', '', regex=False).str.strip()

# 3. Forzar a número entero (los errores o nulos se vuelven NaN)
df['rut_int'] = pd.to_numeric(df['rut_base'], errors='coerce')
ct['rut_int'] = pd.to_numeric(ct['rut_base'], errors='coerce')

print("RUTs estandarizados.")

Estandarizando RUTs al formato numérico base...
RUTs estandarizados.


In [3]:
# %%
print("Calculando RUTs en común...")

# Crear conjuntos matemáticos de RUTs únicos, descartando los vacíos (NaN)
ruts_df = set(df['rut_int'].dropna())
ruts_ct = set(ct['rut_int'].dropna())

# Calcular la intersección
ruts_comunes = ruts_df.intersection(ruts_ct)

print(f"RUTs únicos en Clientes Libres: {len(ruts_df)}")
print(f"RUTs únicos en Cupratherm: {len(ruts_ct)}")
print(f"-> RUTs EN COMÚN (MATCH EXACTO): {len(ruts_comunes)}")

# Si hay coincidencias, imprimimos cuáles son
if ruts_comunes:
    print("\nLista de RUTs coincidentes (cuerpo numérico sin DV):")
    print(sorted(list(ruts_comunes)))
else:
    print("\nNo hay ningún RUT en común entre ambas bases.")

Calculando RUTs en común...
RUTs únicos en Clientes Libres: 73
RUTs únicos en Cupratherm: 677
-> RUTs EN COMÚN (MATCH EXACTO): 2

Lista de RUTs coincidentes (cuerpo numérico sin DV):
[76858530, 87756500]


In [4]:
df_filtrado = df[df['rut_int'].isin(ruts_comunes)]

ct_filtrado = ct[ct["rut_int"].isin(ruts_comunes)]


In [7]:
df_filtrado.drop_duplicates(subset=["clave", "RUT"])

,clave,Zona,Hora,active_calendar,RUT,rut_log,n_ruts,Razon_Social,razon_social_log,n_razones_sociales,...,valorizado_CLP_mean,valorizado_CLP_std,valorizado_CLP_count,medida_total,region,macrozona,confianza,IA,rut_base,rut_int
14256,15832215198RC,Norte,0,111111111111,76.858.530-K,76.858.530-K,1,Noracid S.A.,Noracid S.A.,1,...,-167167.865840,8.233482e+04,365,-63784.130807,Antofagasta,Norte Grande,100.00,False,76858530,76858530
69840,ACF1_VIC,Norte,0,111111111111,76.858.530-K,76.858.530-K,1,Noracid S.A.,Noracid S.A.,1,...,-106482.389414,5.084306e+04,365,-39117.413629,NaN,Norte Grande,99.05,True,76858530,76858530
69864,ACF2_VIC,Norte,0,111111111111,76.858.530-K,76.858.530-K,1,Noracid S.A.,Noracid S.A.,1,...,-47187.668485,2.485750e+04,365,-16256.328362,NaN,Norte Grande,99.05,True,76858530,76858530
93648,ENAPBIO,Sur,0,111111111111,87.756.500-9,87.756.500-9,1,ENAP REFINERIAS S.A. - Aconcagua,ENAP Refinerías S.A.::ENAP REFINERIAS S.A. - A...,2,...,-776395.648773,1.239468e+06,365,-303049.532535,NaN,Centro Sur,85.45,True,87756500,87756500
93696,ENAP_TOR,Norte,0,111111111111,87.756.500-9,87.756.500-9,1,ENAP REFINERIAS S.A. - Aconcagua,ENAP Refinerías S.A.::ENAP REFINERIAS S.A. - A...,2,...,-284952.850702,1.052300e+06,365,-84501.573271,Valparaíso,Centro,100.00,False,87756500,87756500


In [21]:
df_filtrado.columns

Index(['clave', 'Zona', 'Hora', 'active_calendar', 'RUT', 'rut_log', 'n_ruts',
       'Razon_Social', 'razon_social_log', 'n_razones_sociales',
       'Nombre_Corto', 'nombre_corto_log', 'n_nombres_cortos', 'nombre_barra',
       'nombre_barra_log', 'n_nombres_barra', 'tension', 'tension_log',
       'n_tensiones', 'tipo', 'period', 'medida_mean', 'medida_std',
       'CMg[CLP/KWh]_mean', 'CMg[CLP/KWh]_std', 'CMg[CLP/KWh]_count',
       'valorizado_CLP_mean', 'valorizado_CLP_std', 'valorizado_CLP_count',
       'medida_total', 'region', 'macrozona', 'confianza', 'IA', 'rut_base',
       'rut_int'],
      dtype='str')

In [22]:
ct_filtrado[["RAZON_SOCIAL" ,"RUT_RAZON_SOCIAL","NOMBRE_ESTABLECIMIENTO","COMBUSTIBLE_PRIMARIO", "RUBRO", "REGION", "DEMANDA_CALOR_MWH"]].head()

,RAZON_SOCIAL,RUT_RAZON_SOCIAL,NOMBRE_ESTABLECIMIENTO,COMBUSTIBLE_PRIMARIO,RUBRO,REGION,DEMANDA_CALOR_MWH
295,ENAP REFINERIAS S A,87756500-9,PLANTA CO GENERADORA,Gas Natural,Otras actividades,Valparaíso,16048.972820
524,NORACID SA,76858530-K,PLANTA DE ACIDO SULFURICO MEJILLONES,Petróleo N 2 (Diesel),"Industria química, de plástico y caucho",Antofagasta,470.614704
